# Module 7: Finding Failure Modes

> Self-directed module, ~30 min. LangSmith UI focused — no code cells in this notebook.

By this point in the module sequence, your LangSmith project has accumulated a trace corpus — from Module 2's warm-up, Module 3's experiment runs, Module 4's trigger prompts, Module 5's queue triggers, and (if you ran it) Module 6's Studio testing. This module covers how to turn that raw trace data into signal that drives agent improvements — without manually reading every trace.


## The shift in scale

Manually reviewing traces was a viable workflow when agents were short and few. Three trends have pushed past that point:

- **Agents are longer-running.** A single trace can span dozens of tool calls and multiple subagent invocations. Even a careful reviewer needs minutes per trace; scanning hundreds is impractical.
- **Multimodal inputs and outputs are becoming common.** Traces include images, audio, attachments, and structured documents alongside text. Skimming for a pattern means inspecting more than text spans.
- **Agents are proliferating.** A typical enterprise deployment runs many agents, each producing its own trace stream. Even with focused attention, surface area outpaces a single reviewer.

LangSmith ships three features built around this problem. Each turns a different slice of trace data into action:

- **Chat** — a workspace AI assistant. Ask natural-language questions, get answers grounded in your project's traces, threads, prompts, and experiments.
- **Insights Agent** — an automated job that examines a trace corpus and produces hierarchical category reports. Surfaces patterns nobody knew to look for.
- **Engine** — a closed-loop system that detects recurring failures, diagnoses root causes, and proposes fixes as pull requests. Currently available on cloud; self-hosted support is on the roadmap.


---

## Chat

Chat is an AI assistant embedded directly in the LangSmith workspace. It lives in the bottom-right corner of every observability, prompt engineering, and evaluation page and answers natural-language questions about whatever you're currently looking at — traces, threads, prompts, experiments, datasets, queues.

Chat accesses the same data the UI does. The value isn't new visibility — it's skipping the steps of writing a filter expression, scrolling a long list, or opening several traces to compare them. For ad-hoc questions during development or review, it's the fastest path.

<img src="../images/chat.png" alt="Chat assistant panel open in the LangSmith UI" style="width: auto; max-height: 380px; border-radius: 8px;">


### Accessing Chat

- **Keyboard shortcut:** `Cmd+I` (Mac) or `Ctrl+I` (Linux / Windows) — toggles the panel from any page.
- **Clear thread:** `Cmd+Shift+O` / `Ctrl+Shift+O` — starts a fresh conversation.
- **Click** the Chat icon in the bottom-right corner.

**Setup:** Chat needs a model provider API key configured as a workspace secret in **Settings**. Supported providers include Anthropic, OpenAI, Google Gemini, AWS Bedrock, Groq, Mistral, xAI, DeepSeek, and Fireworks.


### Questions worth asking against the client research project

After the Studio testing in Module 6, try these in Chat. Each one assumes you're on a specific LangSmith page — open that page first, then ask Chat the question.

- **On the tracing project page** (the one with your warm-up + Studio traces — open any individual trace first): *"Summarize the tool calls that the agent made in this trace."* — fast way to read a multi-step trace without expanding every span.
- **On the tracing project page, after Module 4 has set up the correctness evaluator:** *"Which runs failed the correctness eval? Summarize why."* — Chat reads the judge's comment field and clusters the failures. Skip this one if you haven't run Module 4.
- **On the tracing project page:** *"Did any users get an unknown-client response?"* — surfaces the `globex-corp` query and any similar lookups.
- **On the tracing project page:** *"What's the latency distribution for runs that delegated to the research subagent?"* — answers questions that would otherwise require export + analysis.
- **On the dataset's Experiments tab (Module 3):** *"Compare this experiment to the previous one. What changed?"* — the page context scopes Chat to the experiments you're looking at.

Chat can also edit prompts conversationally (e.g., *"make the judge prompt stricter on citation requirements"*) — useful in the Prompt Hub. The conversational surface stays consistent; what changes is the page you're on, which scopes what Chat sees.


---

## Insights Agent

Insights is an automated analysis job that runs against a project's trace corpus and groups the runs into a hierarchy of categories — top-level themes (e.g., "requests about specific clients", "market research", "unknown identifiers") with nested subcategories. The output is an executive summary, distribution bars showing how many runs fell into each category, and per-category aggregates (latency, error rate, cost, evaluator feedback).

This is the right tool for the question "what is actually happening in production?" Patterns nobody thought to look for surface as their own categories. Use Insights to find what's worth investigating; use Chat to dig into specific traces once you know where to look.


### Prerequisites

- **A configured model** in the workspace settings (the analysis itself runs an LLM over the traces).
- **A healthy trace corpus.** By this point in the modules, the warm-up in Module 2, the experiments in Module 3, the trigger prompts in Modules 4 and 5, and (optionally) the Studio testing in Module 6 will have collectively produced enough trace volume for Insights to find patterns. Production deployments accrue traces continuously and don't need a manual seed.


### Configuring a report

1. Open the **Tracing Projects** page and select the project (`langsmith-poc` if you kept the default).
2. Click **+New → New Insights Report**.
3. Use the defaults unless you have a specific configuration you want to test — the report will auto-generate a category structure from your trace data.
4. Launch. The job runs in the background and typically completes within 30 minutes.

<img src="../images/insights_config.png" alt="Insights Report configuration dialog" style="width: auto; max-height: 380px; border-radius: 8px;">


### Reading the report

The report opens to an executive summary — the largest patterns by run share, with one-sentence descriptions. From there:

- **Click a category** to drill into its subcategories and individual sample traces.
- **Distribution bars** show how many runs fell into each category, sorted by frequency.
- **Per-category aggregates** show error rate, latency percentiles, cost, and any attached evaluator feedback. A category with high frequency *and* high error rate is the first place to look for a regression.
- **Sample traces** are linked directly — clicking opens the trace in the standard trace tree view.

<img src="../images/insights_report.png" alt="Insights report with hierarchical categories and distribution bars" style="width: auto; max-height: 420px; border-radius: 8px;">


### Scheduling

Reports can be set to re-run on a schedule — daily, weekly, or a custom cron expression. Use the defaults unless you have a specific cadence in mind.


### Acting on Insights findings

Insights surfaces patterns; the loop closes via the earlier modules. When a category looks actionable:

- **High-frequency failure category** → promote sample traces to an annotation queue (Module 5) for human labeling. The labeled examples become eval data for the next iteration.
- **Latency outlier category** → open a sample trace and use **Chat** to summarize the trace tree, then tune the offending tool or prompt.
- **Missing-citation pattern** → tighten the judge prompt (Module 4) to score citation presence more strictly.


---

## Engine

> ### Available on cloud
>
> Engine is **currently available on LangSmith cloud**. Self-hosted support is on the roadmap. If you're running self-hosted, review this section for context — return to Chat and Insights as the failure-mode discovery surface available to you today.


### What Engine does

Engine plugs into the full agent engineering lifecycle: trace → recurring failure detected → root cause diagnosed → fix proposed → evaluator deployed → dataset examples generated. It runs continuously against the project's traces and surfaces issues without requiring manual review.

Where Insights gives you a categorized view of what's happening, Engine produces specific, actionable issues with proposed fixes — code or prompt modifications you can apply via a pull request, plus suggested evaluators to catch regressions on the same failure mode in future iterations. It also generates dataset examples with assertions that you can promote into experiments (Module 5), closing the loop back into your regression suite. Insights surfaces *what's happening*; Engine surfaces *what needs to change*.


### Enabling Engine

Two configuration steps:

1. **Organization Admin** enables Engine workspace-wide in org settings.
2. **Per project**, configure:
   - Optionally connect a **GitHub repository** so fix proposals can become pull requests directly.
   - Select **priority categories** — which failure types matter most.
   - Review the **auto-generated agent overview** document Engine produces; correct any misunderstandings before issue generation starts.

Once enabled, Engine populates an **Issues** list in the project as it identifies failure patterns.

<img src="../images/engine_config.png" alt="Engine configuration dialog with GitHub connection and priority categories" style="width: auto; max-height: 380px; border-radius: 8px;">


### The closed loop

Each Engine issue includes four components:

- **Diagnosis** — the root cause, linked to the failing traces. Inspect the linked traces directly to confirm the diagnosis matches your read.
- **Proposed Fix** — code or prompt changes. If a GitHub repo is connected, openable as a PR with one click.
- **Suggested Evaluator** — a check you can deploy as a new run rule (Module 4) to catch regressions on this failure mode going forward.
- **Add offline examples** — extract the failing runs as dataset entries for the experiment loop from Module 3.

<img src="../images/engine_issue.png" alt="Engine issue detail with diagnosis, proposed fix, and suggested evaluator" style="width: auto; max-height: 420px; border-radius: 8px;">
<img src="../images/engine_triage.png" alt="Engine main triage issue view" style="width: auto; max-height: 420px; border-radius: 8px;">

Priority adjustments and feedback (accept / reject / refine) train Engine's understanding of which issues matter for this specific project — relevance improves over iterations.


---

## Putting it together

A practical workflow for an enterprise POC:

1. **Generate trace volume.** Trace volume accrues across Modules 2–5 as you run the warm-ups, experiments, and trigger prompts. Scripted batch runs or real production traffic work too. Insights and Engine both need enough traces to find patterns.
2. **Use Chat during development and review** for ad-hoc questions — slowest runs this morning, what failed correctness, what changed between experiments. Low setup, immediate.
3. **Configure an Insights report** to surface patterns across the trace corpus. Review the executive summary to catch behavioral drift.
4. **When an Insights category looks actionable** — high-frequency failures, latency outliers, missed citations — promote affected traces to an annotation queue (Module 5) or a dataset (Module 3). The labeled / curated examples drive the next iteration of the judge, the agent, or both.
5. **Engine short-circuits steps 3–4** for recurring failures: it surfaces the issue with a proposed fix, a suggested evaluator, and dataset examples with assertions ready for experiments.


## Recap

| Tool | When to use | Self-hosted |
|---|---|---|
| **Chat** | Ad-hoc questions during development or review. Lowest setup. | Confirm with workspace admin. |
| **Insights Agent** | Automated periodic discovery of patterns across a trace corpus. | Yes. |
| **Engine** | Closed-loop detection, fix proposal, and dataset generation. | Coming soon. |

All three convert raw trace data into signal worth acting on. That's the separator from an incumbent observability platform that stops at "here are the traces" — LangSmith's failure-mode discovery layer is built specifically for the scale agent applications now operate at.

**End of the module sequence.** Modules 02–07 form a continuous loop: traces in → online scoring → human-in-the-loop on flagged runs → offline experiments → deployment → automated pattern surfacing → back to dataset and judge iteration.
